# ハンズオン1: 「いちばん良い組み合わせ」を探す

## 今回の問題: チーム分け

4人を2つのチームに分けます。ただし、**仲が悪い人どうしは別のチームにしたい**とします。仲の悪さには強さがあり、点数で表されています。

ルールはこれだけです。

> 「別々のチームになった、仲の悪いペア」の点数を合計する。この合計がいちばん大きくなる分け方を見つける。

この問題には **Max-Cut（マックスカット）** という名前がついています。数学やコンピュータ科学ではとても有名な問題です。

### どうしてこんな問題が重要なの？

一見すると、ただのパズルに見えます。しかし次のような問題は、実は**まったく同じ構造**をしています。

- トラックの配送ルートを決める
- 工場で作業の順番を決める
- パーティーの座席を決める
- 広告の予算配分を決める
- 電波の周波数を割り当てる

共通点は「**たくさんの選択肢の中から、いちばん良い組み合わせを選ぶ**」こと。これを**組合せ最適化**と呼び、量子コンピュータの有力な応用候補とされています。

### 今日のゴール

量子コンピュータで解くこと自体ではありません。その手前にある、もっと大事な工程です。

> **問題を、最適化マシンが受け取れる形に整え、その変換が正しいことを確かめる**

ここを間違えると、どんなに高性能な機械でも「間違った問題」を一生懸命解いてしまいます。

### 補足: 昔あったサービスについて

以前Azureには最適化専用のサービス（QIO）がありましたが、2023年6月に終了しました。そのため本教材では特定のサービスに頼らず、**どんな実行先にも持っていける形**を学びます。

### 次のセルについて

問題のデータを用意します。`(0, 1, 1.0)` は「**0番さんと1番さんの仲が悪く、その強さは1.0**」という意味です。全部で4人（0番〜3番）、仲の悪いペアが5組あります。

In [ ]:
import itertools
import math

# (頂点i, 頂点j, 重み)
edges = [
    (0, 1, 1.0),
    (0, 2, 2.0),
    (1, 2, 1.5),
    (1, 3, 2.5),
    (2, 3, 1.0),
]
n_vertices = 4

## 1. まず「点数の付け方」を決める

### 変数の意味

各メンバーに `0` か `1` の番号をつけます。

- `0` … Aチーム
- `1` … Bチーム

たとえば `(1, 0, 0, 1)` なら「0番さんと3番さんがBチーム、1番さんと2番さんがAチーム」という意味です。

### いつ点数が入るか

仲の悪いペアが**違うチームに分かれたとき**だけ、そのペアの点数が入ります。コードでは `bits[i] != bits[j]` の部分がこれにあたります。`!=` は「等しくない」という意味です。

同じチームになってしまったら、点数は入りません（けんかが起きてしまうので）。

### 次のセルがすること

4人を2チームに分ける方法は、全部で16通り（2×2×2×2）しかありません。少ないので**全部試して**、いちばん点数が高い分け方を見つけます。

`itertools.product((0, 1), repeat=4)` は「0と1を4つ並べた組み合わせを全部作る」という意味で、`0000`、`0001`、`0010`、… `1111` と16通りを順に生成します。

この「全部試した答え」は、あとで**正解の基準**として使います。

In [ ]:
def cut_value(bits, weighted_edges):
    return sum(weight for i, j, weight in weighted_edges if bits[i] != bits[j])

candidates = []
for bits in itertools.product((0, 1), repeat=n_vertices):
    candidates.append((cut_value(bits, edges), bits))

best_value, best_bits = max(candidates)
print("best assignment:", best_bits)
print("best cut value:", best_value)

## 2. 最適化マシンが読める書式に直す（QUBO）

### なぜ変換が必要なのか

レストランで注文するとき、好きな文章で頼んでも通じません。メニューの形式に合わせる必要があります。最適化マシンも同じで、**決まった書式**でしか問題を受け取れません。

その代表的な書式が **QUBO**（キューボ）です。

### QUBOとは

難しそうな名前ですが、中身はシンプルです。

> **0か1の変数だけを使った、掛け算が2つまでの式**

「2つまで」が重要です。`x1 × x2` はOK。`x1 × x2 × x3` のように3つ掛けるのはダメです。この制限があるおかげで、機械は効率よく計算できます。

### 「違うチームなら1」を式で書く

ここが今回の山場ですが、表を見れば納得できます。使う式は次のものです。

`x_i + x_j - 2 × x_i × x_j`

| x_i | x_j | 式の計算 | 結果 | 違うチーム？ |
|---|---|---|---|---|
| 0 | 0 | 0 + 0 - 0 | **0** | 同じ（点数なし） |
| 0 | 1 | 0 + 1 - 0 | **1** | 違う（点数あり） |
| 1 | 0 | 1 + 0 - 0 | **1** | 違う（点数あり） |
| 1 | 1 | 1 + 1 - 2 | **0** | 同じ（点数なし） |

違うチームのときだけ `1` になっています。しかも掛け算は2つまでです。つまり**QUBOの条件を満たしています**。

### 「大きくしたい」を「小さくしたい」に変える

QUBOの機械は基本的に「**値をできるだけ小さくする**」ように作られています。しかし今回は点数を**大きく**したい。

そこで、全体にマイナスを掛けて向きを逆にします。「得点を最大にする」は「マイナスの得点を最小にする」と同じことだからです。気温と同じで、「暖かさを最大に」と「寒さを最小に」は同じ意味です。

この結果、あとで出てくる答えは `-6.5` のようにマイナスになります。符号を外した `6.5` が本来の得点です。

### 次のセルがすること

変換後の係数を辞書の形で作ります。`(0, 0)` のように同じ番号が並んでいるものは `x_0 × x_0` の項ですが、`0×0=0`、`1×1=1` なので、これは `x_0` そのものと同じです。

In [ ]:
def maxcut_qubo(weighted_edges):
    qubo = {}
    for i, j, weight in weighted_edges:
        qubo[(i, i)] = qubo.get((i, i), 0.0) - weight
        qubo[(j, j)] = qubo.get((j, j), 0.0) - weight
        key = (min(i, j), max(i, j))
        qubo[key] = qubo.get(key, 0.0) + 2.0 * weight
    return qubo

def qubo_energy(bits, qubo):
    return sum(coefficient * bits[i] * bits[j] for (i, j), coefficient in qubo.items())

qubo = maxcut_qubo(edges)
for key, coefficient in sorted(qubo.items()):
    print(f"Q{key} = {coefficient:+.1f}")

## 3. 変換が正しいかを、必ず確かめる

**この節が、このノートブックで最も実務的に重要な部分です。**

### なぜ確認が必要なのか

さきほどの変換には、間違えやすい落とし穴がいくつもあります。

- プラスとマイナスを逆にしてしまう
- 同じペアを2回数えてしまう、または1回しか数えない
- 「最大化」と「最小化」を取り違える

しかも厄介なことに、**変換を間違えても機械はエラーを出しません**。黙って「間違った問題」の正解を返してきます。そして返ってきた答えは、もっともらしく見えてしまいます。

これは翻訳の誤りに似ています。誤訳された契約書でも、文章としては立派に読めてしまうのと同じです。

### 次のセルがすること

16通りの分け方すべてについて、

- 元のルールで計算した点数
- QUBOに変換した式で計算した値

の2つを比べ、きちんと符号が逆の関係になっているかを確認します。

`assert` は「もし違っていたら、そこで止めて知らせる」命令です。**何も表示されずに通過すれば合格**です。静かに通ることが良い知らせ、という少し変わった仕組みです。

最後に、QUBO側で見つけた最良の答えが、最初に全部試して見つけた答えと一致することも確認します。

In [ ]:
for bits in itertools.product((0, 1), repeat=n_vertices):
    original = cut_value(bits, edges)
    transformed = qubo_energy(bits, qubo)
    assert math.isclose(transformed, -original), (bits, original, transformed)

qubo_best_energy, qubo_best_bits = min(
    (qubo_energy(bits, qubo), bits)
    for bits in itertools.product((0, 1), repeat=n_vertices)
)
print("QUBO best:", qubo_best_bits, "energy:", qubo_best_energy)
assert math.isclose(-qubo_best_energy, best_value)

## 4. なぜ「全部試す」では済まないのか

今回は4人だったので、16通りを全部試せました。では人数が増えるとどうなるでしょうか。

### 組み合わせの数は爆発する

n人いると、分け方は `2のn乗` 通りあります。次のセルで実際の数字を見てください。

- 10人 … 1,024通り（一瞬）
- 20人 … 約100万通り（まだ平気）
- 30人 … 約10億通り（そろそろ厳しい）
- 50人 … **約1125兆通り**

50人の場合、1秒間に10億通り調べられる高速なコンピュータを使っても、**約13日**かかります。60人なら約36年です。

人数が10増えるたびに、かかる時間が約1000倍になっていきます。これが**組合せ爆発**と呼ばれる現象です。

### 現実の問題はもっと大きい

実際の配送計画や生産計画では、変数が数百から数万個になることも珍しくありません。全部試すことは永遠に不可能です。

だからこそ賢い解き方が必要で、量子コンピュータへの期待もここにあります。

### ただし、フェアな比較を忘れずに

ここで注意があります。量子コンピュータの性能を語るとき、比較相手を「全部試す方法」にしてはいけません。それは**最も素朴なやり方**だからです。

人類はこの数十年で、全部試さずに良い答えを見つける優れた古典アルゴリズムをたくさん開発してきました。公平な比較のためには、そうした実力のある手法と比べる必要があります。

In [ ]:
for n in (4, 10, 20, 30, 50):
    print(f"n={n:>2}: candidates={2 ** n:,}")

## 5. 実際に量子マシンへ渡すときのチェックリスト

作ったQUBOを実行先に持っていくときの確認事項です。今すぐ全部を理解する必要はありません。「こういう注意点があるんだな」と眺めてください。

### 量子アニーリング方式に渡す場合

- 係数の大きさが、機械が扱える範囲に収まっているか
- 問題のつながり方を、機械の配線にうまく載せられるか（**埋め込み**と呼びます）
- ルールを守らせるための罰金の強さは適切か

### ゲート方式（QAOAなど）に渡す場合

- QUBOを量子回路の形に書き換える
- 回路の長さ（深さ）が、機械のノイズに耐えられる範囲か
- 古典コンピュータとの往復回数は現実的か

### Azure Quantumで実行する場合

- `workspace.get_targets()` で、**今使える機械**を必ず確認する（一覧は変わります）
- 選んだ機械が、その入力形式を受け付けるか確認する
- 終了したサービス（旧QIO）向けの古い記事のコードを使わない

### 結果の評価で見るべきこと

「動いた」で満足しないことが大切です。次の5点を記録しましょう。

1. **解の良さ** … 正解とどれだけ近いか
2. **ルール違反の割合** … 制約を守れなかった回数
3. **かかった時間** … 順番待ちを含めた合計
4. **費用** … 実行回数に応じた料金
5. **再現性** … もう一度やって同じ結果になるか

## 演習

手を動かすと理解が定着します。コードを書き換えて実行してみてください。

### 演習1: 新しい人間関係を追加する

`edges` のリストに `(0, 3, 3.0)` を追加してください。「0番さんと3番さんも仲が悪く、その強さは3.0」という意味です。実行して、最適な分け方がどう変わるか確認しましょう。

**考えるポイント:** 強い対立が増えると、チーム分けはどう変わるでしょうか。

### 演習2: 条件を固定する

「0番さんは必ずAチーム（`0`）にする」という条件を加えてください。

**ヒント:** 全部試すループの中で、`bits[0] == 0` のものだけを対象にすれば実現できます。

### 演習3: 規模を大きくしてみる

10人分のランダムなデータを作り、全部試すのにどれくらい時間がかかるか測ってください。

**ヒント:** `import time` を使い、処理の前後で `time.time()` の差を取ります。

### 演習4: 言葉で説明する

QUBO変換の検証（第3節）が、なぜ量子マシンに送る前に必要なのでしょうか。プログラミングを知らない上司に説明するつもりで、3行程度にまとめてください。